# LeetCode #164 — Maximum Gap: chia ngăn (bucket) và nguyên lý chuồng bồ câu

**Mức độ:** Medium · **Chủ đề:** Array, Sorting, Bucket Sort, Radix Sort

Skeleton trên LeetCode:

```python
class Solution:
    def maximumGap(self, nums: List[int]) -> int:
```

Mục tiêu của notebook:

1. Lời giải dễ: **sắp xếp** rồi lấy hiệu lớn nhất giữa hai phần tử liền kề — `O(n log n)`.
2. Đề yêu cầu **thời gian và bộ nhớ tuyến tính** → cần cách không dùng sắp xếp so sánh.
3. **Chia ngăn** theo giá trị + **nguyên lý chuồng bồ câu**: khoảng cách lớn nhất không thể nằm **trong** một ngăn, nên chỉ cần `min`/`max` của từng ngăn.
4. Cách khác: **radix sort** (sắp xếp theo từng chữ số).

# 1. Bài toán là gì?

Cho mảng số nguyên `nums` (chưa sắp xếp). Trả về **hiệu lớn nhất** giữa hai phần tử **liền kề** trong dạng **đã sắp xếp** của nó. Nếu mảng có ít hơn 2 phần tử, trả về `0`.

Bạn phải viết thuật toán chạy trong **thời gian tuyến tính** và dùng **bộ nhớ phụ tuyến tính**.

**Constraints (theo đề):**

- `1 <= nums.length <= 10^5`
- `0 <= nums[i] <= 10^9`

---

## Ví dụ

```text
[3, 6, 9, 1]   ->  3     (sắp xếp: [1, 3, 6, 9], các hiệu 2, 3, 3)
[10]           ->  0
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Sắp xếp (sorting)
2. Sắp xếp đếm, sắp xếp theo xô (bucket) và nguyên lý chuồng bồ câu

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Sắp xếp (sorting)

### Sắp xếp trong Python

- `sorted(a)` trả về **list mới**; `a.sort()` sắp xếp **tại chỗ** (trả về `None`).
- `key=` chọn tiêu chí: `sorted(words, key=len)`, `sorted(pairs, key=lambda p: p[1])`.
- `reverse=True` để giảm dần; sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple.
- Thuật toán bên trong là **Timsort**: `O(n log n)`, **ổn định** (stable — hai phần tử bằng nhau giữ nguyên thứ tự ban đầu).

In [1]:
a_sort = [5, 2, 9, 1, 5, 6]
print("sorted:", sorted(a_sort), "| gốc không đổi:", a_sort)
a_sort.sort(reverse=True)
print("sort giảm dần tại chỗ:", a_sort)

nguoi = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print("theo tuổi (ổn định — An trước Chi):", sorted(nguoi, key=lambda p: p[1]))
print("tuổi giảm, rồi tên tăng:", sorted(nguoi, key=lambda p: (-p[1], p[0])))
print("chuỗi sắp theo ký tự:", "".join(sorted("banana")))

sorted: [1, 2, 5, 5, 6, 9] | gốc không đổi: [5, 2, 9, 1, 5, 6]
sort giảm dần tại chỗ: [9, 6, 5, 5, 2, 1]
theo tuổi (ổn định — An trước Chi): [('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
tuổi giảm, rồi tên tăng: [('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]
chuỗi sắp theo ký tự: aaabnn


### Vì sao sắp xếp hay là bước đầu tiên?

Sau khi sắp xếp: phần tử **bằng nhau đứng cạnh nhau** (dễ bỏ trùng), có thể dùng **hai con trỏ** / **tìm nhị phân**, các khoảng chồng nhau **liền kề** nhau. Đổi lại tốn `O(n log n)`.

### Sắp xếp chèn (insertion sort)

Giống xếp bài trên tay: lấy lần lượt từng phần tử, **chèn** vào đúng vị trí trong phần đã sắp xếp bên trái. `O(n²)` xấu nhất, nhưng `O(n)` khi dữ liệu gần như đã sắp xếp.

In [2]:
def sap_xep_chen(a):
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]            # dịch phần tử lớn hơn sang phải
            j -= 1
        a[j + 1] = x
        print(f"   sau khi chèn {x}: {a[:i + 1]} | {a[i + 1:]}")
    return a

print("→", sap_xep_chen([5, 2, 4, 6, 1, 3]))

   sau khi chèn 2: [2, 5] | [4, 6, 1, 3]
   sau khi chèn 4: [2, 4, 5] | [6, 1, 3]
   sau khi chèn 6: [2, 4, 5, 6] | [1, 3]
   sau khi chèn 1: [1, 2, 4, 5, 6] | [3]
   sau khi chèn 3: [1, 2, 3, 4, 5, 6] | []
→ [1, 2, 3, 4, 5, 6]


### Sắp xếp trộn (merge sort) và phép trộn hai dãy đã sắp xếp

**Chia để trị**: chia đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp. Trộn: hai con trỏ ở đầu hai dãy, mỗi bước lấy phần tử nhỏ hơn. `O(n log n)` mọi trường hợp, ổn định (nhờ lấy bên trái khi bằng nhau).

```text
[5,2,4,6,1,3] → [5,2,4] [6,1,3] → [5] [2,4] [6] [1,3] → … → [2,4,5] [1,3,6] → [1,2,3,4,5,6]
```

In [3]:
def tron(x, y):
    i = j = 0
    out = []
    while i < len(x) and j < len(y):
        if x[i] <= y[j]:               # <= giữ ổn định
            out.append(x[i]); i += 1
        else:
            out.append(y[j]); j += 1
    out.extend(x[i:]); out.extend(y[j:])
    return out

def sap_xep_tron(a, sau=0):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    trai, phai = sap_xep_tron(a[:mid], sau + 1), sap_xep_tron(a[mid:], sau + 1)
    kq = tron(trai, phai)
    print("    " * sau + f"trộn {trai} + {phai} = {kq}")
    return kq

print("→", sap_xep_tron([5, 2, 4, 6, 1, 3]))

        trộn [2] + [4] = [2, 4]
    trộn [5] + [2, 4] = [2, 4, 5]
        trộn [1] + [3] = [1, 3]
    trộn [6] + [1, 3] = [1, 3, 6]
trộn [2, 4, 5] + [1, 3, 6] = [1, 2, 3, 4, 5, 6]
→ [1, 2, 3, 4, 5, 6]


### Bảng so sánh

| Thuật toán | Tốt nhất | Xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|---|
| Insertion sort | `O(n)` | `O(n²)` | `O(1)` | có |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n)` (mảng) | có |
| Quick sort | `O(n log n)` | `O(n²)` | `O(log n)` | không |
| Timsort (`sorted`) | `O(n)` | `O(n log n)` | `O(n)` | có |

Lỗi hay gặp: `a = a.sort()` (gán `None`); quên rằng `sorted` tạo list mới (tốn bộ nhớ); `key` trả về kiểu không so sánh được.

## 2.2 Sắp xếp đếm, sắp xếp theo xô (bucket) và nguyên lý chuồng bồ câu

### Vượt qua giới hạn O(n log n)

Mọi thuật toán sắp xếp **chỉ dựa vào so sánh** cần `Ω(n log n)` phép so sánh. Nếu biết thêm về **giá trị** (số nguyên trong khoảng nhỏ, phân bố đều…), ta có thể làm `O(n)` bằng cách **phân phối** phần tử vào các ô theo giá trị.

### Sắp xếp đếm (counting sort)

Giá trị nguyên trong `[0, K)`: đếm số lần xuất hiện mỗi giá trị, rồi in ra theo thứ tự. `O(n + K)`.

In [4]:
def counting_sort_demo(a, K):
    cnt = [0] * K
    for x in a:
        cnt[x] += 1
    out = []
    for v in range(K):
        out.extend([v] * cnt[v])
    return out

print(counting_sort_demo([3, 1, 4, 1, 5, 9, 2, 6, 5, 3], 10))

[1, 1, 2, 3, 3, 4, 5, 5, 6, 9]


### Sắp xếp theo xô (bucket sort)

Chia khoảng `[lo, hi]` thành `B` xô bằng nhau; phần tử `x` rơi vào xô `(x − lo) * B // (hi − lo + 1)`. Sắp từng xô (nhỏ) rồi nối lại. Nếu dữ liệu phân bố đều, mỗi xô có `O(1)` phần tử → tổng `O(n)` trung bình.

In [5]:
def bucket_sort_demo(a, B):
    lo, hi = min(a), max(a)
    buckets = [[] for _ in range(B)]
    for x in a:
        buckets[(x - lo) * B // (hi - lo + 1)].append(x)
    for i, b in enumerate(buckets):
        print(f"   xô {i}: {b}")
    return [x for b in buckets for x in sorted(b)]

print(bucket_sort_demo([29, 3, 47, 15, 8, 33, 41, 22], 4))

   xô 0: [3, 8]
   xô 1: [15, 22]
   xô 2: [29, 33]
   xô 3: [47, 41]
[3, 8, 15, 22, 29, 33, 41, 47]


### Nguyên lý chuồng bồ câu (pigeonhole principle)

Nhốt `n + 1` con bồ câu vào `n` chuồng → có chuồng chứa **ít nhất 2** con. Phiên bản "khoảng trống": `n` số trong `[lo, hi]` có `n − 1` khoảng cách giữa các số kề nhau (sau khi sắp), tổng các khoảng = `hi − lo` → **khoảng lớn nhất `>= ⌈(hi − lo) / (n − 1)⌉`** (lớn hơn hoặc bằng trung bình).

Hệ quả cho bucket sort: nếu chọn độ rộng xô **nhỏ hơn** khoảng lớn nhất, hai số kề nhau có khoảng lớn nhất **không thể** nằm cùng một xô → chỉ cần so sánh **max của xô trước** với **min của xô sau**.

In [6]:
nums_demo = [1, 3, 9, 4, 20]
s_demo = sorted(nums_demo)
gaps = [b - a for a, b in zip(s_demo, s_demo[1:])]
print("sắp xếp:", s_demo, "| các khoảng:", gaps, "| tổng =", sum(gaps), "= hi − lo =", s_demo[-1] - s_demo[0])
print("khoảng lớn nhất", max(gaps), ">= trung bình", (s_demo[-1] - s_demo[0]) / (len(s_demo) - 1))

sắp xếp: [1, 3, 4, 9, 20] | các khoảng: [2, 1, 5, 11] | tổng = 19 = hi − lo = 19
khoảng lớn nhất 11 >= trung bình 4.75


### Lỗi hay gặp

- Công thức chỉ số xô cho ra `B` (vượt mảng) với `x = hi` → dùng `hi − lo + 1` ở mẫu hoặc thêm một xô.
- Tất cả phần tử bằng nhau (`hi == lo`) → chia cho 0 nếu không xử lý riêng.
- Counting sort với `K` rất lớn (ví dụ `10^9`) → tốn bộ nhớ, không dùng được.

## 2.3 Áp dụng vào bài này

Cách hiển nhiên là **sắp xếp** rồi lấy khoảng lớn nhất giữa hai phần tử kề nhau — `O(n log n)`. Đề đòi `O(n)` nên ta dùng **sắp xếp theo xô** kết hợp **nguyên lý chuồng bồ câu**: khoảng lớn nhất `>= ⌈(max − min)/(n − 1)⌉`, nên nếu chọn độ rộng xô nhỏ hơn mức đó thì hai số tạo khoảng lớn nhất nằm ở **hai xô khác nhau** — chỉ cần lưu min/max mỗi xô và so `min` xô sau với `max` xô trước còn khác rỗng.

# 3. Nền tảng

In [7]:
from typing import List
import random, time

a = [3, 6, 9, 1]
s = sorted(a)
print("sắp xếp:", s, "| hiệu liền kề:", [s[i + 1] - s[i] for i in range(len(s) - 1)])
print("hàm ceil không dùng float: ceil(7/3) =", -(-7 // 3), "| (7 + 3 - 1) // 3 =", (7 + 3 - 1) // 3)

sắp xếp: [1, 3, 6, 9] | hiệu liền kề: [2, 3, 3]
hàm ceil không dùng float: ceil(7/3) = 3 | (7 + 3 - 1) // 3 = 3


`-(-a // b)` là cách làm tròn **lên** phép chia nguyên mà không qua số thực (tránh sai số với số lớn).

# 4. Baseline: sắp xếp

In [8]:
def max_gap_sort(nums: List[int]) -> int:
    if len(nums) < 2:
        return 0
    s = sorted(nums)
    return max(s[i + 1] - s[i] for i in range(len(s) - 1))

print(max_gap_sort([3, 6, 9, 1]), max_gap_sort([10]))

3 0


`O(n log n)` — trong thực tế với `n = 10^5` vẫn rất nhanh, nhưng **không** thỏa yêu cầu tuyến tính của đề. Ta cần hiểu vì sao làm được `O(n)`.

# 5. Ý tưởng chính: chia ngăn theo giá trị

Gọi `lo = min(nums)`, `hi = max(nums)`, `n = len(nums)`. Có `n − 1` khoảng giữa các phần tử liền kề (đã sắp xếp), tổng độ dài là `hi − lo`.

**Quan sát (chuồng bồ câu):** khoảng lớn nhất **ít nhất** bằng trung bình:

```text
max_gap >= ceil((hi - lo) / (n - 1))
```

Chia đoạn `[lo, hi]` thành các **ngăn** có độ rộng `size = max(1, (hi − lo) // (n − 1))` (≤ trung bình). Hai số trong **cùng một ngăn** cách nhau `< size` ≤ khoảng lớn nhất → **khoảng lớn nhất không nằm trong một ngăn**, mà luôn **nối hai ngăn khác nhau**: từ `max` của một ngăn tới `min` của ngăn **khác rỗng kế tiếp**.

Vậy chỉ cần lưu `min` và `max` của mỗi ngăn:

```text
size = max(1, (hi - lo) // (n - 1))
k = (hi - lo) // size + 1                     số ngăn
ngăn của x: (x - lo) // size
đáp án = max qua các ngăn khác rỗng liên tiếp:  min(ngăn sau) - max(ngăn trước)
```

```text
nums = [1, 3, 6, 9, 10, 17], lo = 1, hi = 17, n = 6 → size = 16 // 5 = 3
ngăn 0 [1, 4):  {1, 3}      min 1  max 3
ngăn 1 [4, 7):  {6}         min 6  max 6       6 - 3 = 3
ngăn 2 [7, 10): {9}         min 9  max 9       9 - 6 = 3
ngăn 3 [10,13): {10}        min 10 max 10     10 - 9 = 1
ngăn 4 [13,16): rỗng
ngăn 5 [16,19): {17}        min 17 max 17     17 - 10 = 7  ← lớn nhất
```

In [9]:
def max_gap_bucket(nums: List[int]) -> int:
    n = len(nums)
    if n < 2:
        return 0
    lo, hi = min(nums), max(nums)
    if lo == hi:
        return 0
    size = max(1, (hi - lo) // (n - 1))
    k = (hi - lo) // size + 1
    bmin = [None] * k
    bmax = [None] * k
    for x in nums:
        b = (x - lo) // size
        bmin[b] = x if bmin[b] is None else min(bmin[b], x)
        bmax[b] = x if bmax[b] is None else max(bmax[b], x)
    best, prev_max = 0, None
    for b in range(k):
        if bmin[b] is None:
            continue                          # ngăn rỗng
        if prev_max is not None:
            best = max(best, bmin[b] - prev_max)
        prev_max = bmax[b]
    return best

print(max_gap_bucket([3, 6, 9, 1]), max_gap_bucket([1, 3, 6, 9, 10, 17]), max_gap_bucket([10]))

3 7 0


# 6. Mô phỏng

In [10]:
def trace_bucket(nums):
    n, lo, hi = len(nums), min(nums), max(nums)
    size = max(1, (hi - lo) // (n - 1))
    k = (hi - lo) // size + 1
    print(f"lo = {lo}, hi = {hi}, n = {n} → size = {size}, số ngăn = {k}")
    buckets = [[] for _ in range(k)]
    for x in nums:
        buckets[(x - lo) // size].append(x)
    best, prev_max = 0, None
    for b, items in enumerate(buckets):
        rng = f"[{lo + b * size}, {lo + (b + 1) * size})"
        if not items:
            print(f"   ngăn {b} {rng:<10}: rỗng")
            continue
        gap = "" if prev_max is None else f" | khoảng từ ngăn trước: {min(items)} - {prev_max} = {min(items) - prev_max}"
        if prev_max is not None:
            best = max(best, min(items) - prev_max)
        print(f"   ngăn {b} {rng:<10}: {sorted(items)} min {min(items)} max {max(items)}{gap}")
        prev_max = max(items)
    print("→ khoảng lớn nhất:", best)
    return best

_ = trace_bucket([1, 3, 6, 9, 10, 17])

lo = 1, hi = 17, n = 6 → size = 3, số ngăn = 6
   ngăn 0 [1, 4)    : [1, 3] min 1 max 3
   ngăn 1 [4, 7)    : [6] min 6 max 6 | khoảng từ ngăn trước: 6 - 3 = 3
   ngăn 2 [7, 10)   : [9] min 9 max 9 | khoảng từ ngăn trước: 9 - 6 = 3
   ngăn 3 [10, 13)  : [10] min 10 max 10 | khoảng từ ngăn trước: 10 - 9 = 1
   ngăn 4 [13, 16)  : rỗng
   ngăn 5 [16, 19)  : [17] min 17 max 17 | khoảng từ ngăn trước: 17 - 10 = 7
→ khoảng lớn nhất: 7


# 7. Phiên bản đúng format LeetCode

In [11]:
class Solution:
    def maximumGap(self, nums: List[int]) -> int:
        n = len(nums)
        if n < 2:
            return 0
        lo, hi = min(nums), max(nums)
        if lo == hi:
            return 0
        size = max(1, (hi - lo) // (n - 1))
        k = (hi - lo) // size + 1
        INF = float("inf")
        bmin = [INF] * k
        bmax = [-INF] * k
        for x in nums:
            b = (x - lo) // size
            if x < bmin[b]: bmin[b] = x
            if x > bmax[b]: bmax[b] = x
        best, prev_max = 0, lo                  # ngăn 0 luôn chứa lo
        for b in range(k):
            if bmin[b] == INF:
                continue
            best = max(best, bmin[b] - prev_max)
            prev_max = bmax[b]
        return best

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.maximumGap([3, 6, 9, 1]), 3)
check("Ví dụ 2", sol.maximumGap([10]), 0)
check("Bằng nhau", sol.maximumGap([5, 5, 5]), 0)
check("Hai phần tử", sol.maximumGap([1, 10**9]), 10**9 - 1)
check("Có trùng", sol.maximumGap([1, 1, 1, 5, 5, 9]), 4)

random.seed(164)
for _ in range(5000):
    nums = [random.randint(0, random.choice([5, 50, 10**9])) for _ in range(random.randint(1, 15))]
    assert sol.maximumGap(nums) == max_gap_sort(nums) == max_gap_bucket(nums), nums
print("5000 mảng ngẫu nhiên: khớp cách sắp xếp")

big = [random.randint(0, 10**9) for _ in range(10**5)]
for name, f in [("chia ngăn", sol.maximumGap), ("sắp xếp", max_gap_sort)]:
    t0 = time.time(); r = f(big); t1 = time.time()
    print(f"n = 10^5, {name:<9}: {t1 - t0:.3f}s, kết quả {r}")

Ví dụ 1          PASS  3
Ví dụ 2          PASS  0
Bằng nhau        PASS  0
Hai phần tử      PASS  999999999
Có trùng         PASS  4


5000 mảng ngẫu nhiên: khớp cách sắp xếp

n = 10^5, chia ngăn: 0.171s, kết quả 104143
n = 10^5, sắp xếp  : 0.072s, kết quả 104143


Trên Python, `sorted` viết bằng C nên chạy **nhanh ngang, thậm chí nhanh hơn** vòng lặp chia ngăn viết bằng Python thuần, dù độ phức tạp lý thuyết `O(n log n)` lớn hơn `O(n)`. Bài này chủ yếu kiểm tra bạn có **hiểu** được cách đạt `O(n)` hay không.

# 8. Vì sao đúng? Độ phức tạp?

Gọi `G` là khoảng lớn nhất thật. `n` số nằm trong `[lo, hi]` tạo `n − 1` khoảng có tổng `hi − lo`, nên `G >= (hi − lo) / (n − 1) >= size` (vì `size = ⌊(hi − lo)/(n − 1)⌋`; nếu `size` bị nâng lên 1 thì `G >= 1 = size` vì các số khác nhau ở hai đầu).

Hai số cùng ngăn cách nhau **nhiều nhất** `size − 1 < G`. Vậy cặp liền kề tạo ra `G` không cùng ngăn: số nhỏ là `max` của ngăn chứa nó, số lớn là `min` của ngăn khác rỗng **kế tiếp** (mọi ngăn giữa hai ngăn đó đều rỗng, nếu không đã có số nằm giữa). Thuật toán xét đúng các cặp (`max` ngăn trước, `min` ngăn sau) đó, và mỗi cặp đó cũng là hai số liền kề thật → lấy `max` cho đúng `G`.

```text
Time:  O(n + k) = O(n)        (k = số ngăn ≤ n)
Space: O(n)
```

Số ngăn: `k = (hi − lo) // size + 1`. Vì `⌊x⌋ >= x/2` khi `x >= 1`, ta có `size >= (hi − lo) / (2(n − 1))`, nên `k <= 2(n − 1) + 1` — vẫn `O(n)`.

# 9. Những lỗi dễ mắc

## Lỗi 1: Chỉ so các ngăn **kề nhau** theo chỉ số (không bỏ qua ngăn rỗng)

In [13]:
def wrong_adjacent_buckets(nums):
    n, lo, hi = len(nums), min(nums), max(nums)
    if n < 2 or lo == hi:
        return 0
    size = max(1, (hi - lo) // (n - 1))
    k = (hi - lo) // size + 1
    bmin, bmax = [None] * k, [None] * k
    for x in nums:
        b = (x - lo) // size
        bmin[b] = x if bmin[b] is None else min(bmin[b], x)
        bmax[b] = x if bmax[b] is None else max(bmax[b], x)
    best = 0
    for b in range(1, k):
        if bmin[b] is not None and bmax[b - 1] is not None:   # SAI: ngăn b-1 rỗng thì bỏ mất khoảng
            best = max(best, bmin[b] - bmax[b - 1])
    return best

print("Sai :", wrong_adjacent_buckets([1, 3, 6, 9, 10, 17]), "| Đúng:", sol.maximumGap([1, 3, 6, 9, 10, 17]))

Sai : 3 | Đúng: 7


Ngăn 4 rỗng nên khoảng `17 − 10 = 7` (nối ngăn 3 với ngăn 5) bị bỏ qua. Khoảng lớn **thường đi kèm** ngăn rỗng — phải nối với ngăn **khác rỗng** kế tiếp.

## Lỗi 2: `size = (hi − lo) // (n − 1)` mà không chặn dưới bằng 1

In [14]:
try:
    lo_, hi_, n_ = 1, 3, 5
    size_ = (hi_ - lo_) // (n_ - 1)
    print("size =", size_, "→", (2 - lo_) // size_)
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e, "— với nums như [1, 2, 3, 2, 1], (3 - 1) // 4 = 0")

ZeroDivisionError: integer division or modulo by zero — với nums như [1, 2, 3, 2, 1], (3 - 1) // 4 = 0


## Lỗi 3: Chỉ số ngăn vượt mảng vì số ngăn `k` tính sai

Nếu dùng `k = n − 1` thì `x = hi` có chỉ số `(hi − lo) // size` có thể bằng `n − 1` hoặc hơn → `IndexError`. Công thức an toàn: `k = (hi − lo) // size + 1`.

In [15]:
def wrong_k(nums):
    n, lo, hi = len(nums), min(nums), max(nums)
    size = max(1, (hi - lo) // (n - 1))
    k = n - 1                                      # SAI
    bmin = [None] * k
    for x in nums:
        bmin[(x - lo) // size] = x
    return k

try:
    wrong_k([3, 6, 9, 1])
except IndexError as e:
    print("IndexError:", e)

IndexError: list assignment index out of range


## Lỗi 4: Dùng số thực để tính chỉ số ngăn

`(x - lo) / size` qua `float` với số tới `10^9` thường vẫn đúng, nhưng làm tròn sai ở biên là nguồn lỗi khó tìm. Dùng **chia nguyên** `//`.

# 10. Mẹo nhận diện pattern

- **"Thời gian tuyến tính" cho bài có vẻ cần sắp xếp** → sắp xếp không so sánh: **đếm** (counting), **chia ngăn** (bucket), **theo chữ số** (radix).
- **Khoảng cách lớn nhất / nhỏ nhất theo giá trị** → chia ngăn có độ rộng hợp lý, chỉ giữ `min`/`max` mỗi ngăn.
- **Nguyên lý chuồng bồ câu**: `n` vật vào `m < n` chuồng thì có chuồng chứa ≥ 2 vật; tổng `S` chia cho `k` phần thì có phần ≥ `S/k`.

Bài liên quan: **#220 Contains Duplicate III** (Hard, chia ngăn), **#912 Sort an Array**, **#75 Sort Colors** (đếm), **#347 Top K Frequent Elements** (bucket theo tần suất), **#1365** (đếm).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chia ngăn cho `nums = [12, 3, 25, 7, 40]`: tính `size`, số ngăn, nội dung mỗi ngăn và đáp án.

## Bài 2 — Điền chỗ trống

In [16]:
def exercise_2_maximum_gap(nums: List[int]) -> int:
    n = len(nums)
    if n < 2:
        return 0
    lo, hi = min(nums), max(nums)
    if lo == hi:
        return 0
    size = max(1, (hi - lo) // (n - 1))
    k = (hi - lo) // size + 1
    bmin, bmax = [None] * k, [None] * k
    # TODO: đưa từng số vào ngăn (cập nhật min/max), rồi duyệt các ngăn khác rỗng
    pass


# assert exercise_2_maximum_gap([3, 6, 9, 1]) == 3
# assert exercise_2_maximum_gap([10]) == 0
# assert exercise_2_maximum_gap([1, 3, 6, 9, 10, 17]) == 7

## Bài 3 — Biến thể: radix sort (LSD, cơ số 10)

Sắp xếp các số nguyên không âm bằng cách lần lượt sắp **ổn định** theo chữ số hàng đơn vị, hàng chục, …  Mỗi lượt là một lần phân phối vào 10 ngăn. `O(d·n)` với `d` = số chữ số.

In [17]:
def exercise_3_radix_sort(nums: List[int]) -> List[int]:
    # TODO
    pass


# assert exercise_3_radix_sort([170, 45, 75, 90, 802, 24, 2, 66]) == [2, 24, 45, 66, 75, 90, 170, 802]
# assert exercise_3_radix_sort([]) == []
# assert exercise_3_radix_sort([5, 0, 5]) == [0, 5, 5]

## Bài 4 — Biến thể: counting sort khi giá trị nhỏ

Giá trị trong `[0, m]` với `m` nhỏ: đếm số lần xuất hiện mỗi giá trị rồi ghi lại. `O(n + m)`.

In [18]:
def exercise_4_counting_sort(nums: List[int], m: int) -> List[int]:
    # TODO
    pass


# assert exercise_4_counting_sort([3, 1, 2, 3, 0, 1], 3) == [0, 1, 1, 2, 3, 3]
# assert exercise_4_counting_sort([], 5) == []

## Bài 5 — Mini challenge: Contains Duplicate III (LeetCode #220, Hard)

Có tồn tại hai chỉ số `i != j` với `|i − j| <= indexDiff` và `|nums[i] − nums[j]| <= valueDiff` không? Làm trong `O(n)` bằng chia ngăn độ rộng `valueDiff + 1`, chỉ giữ các phần tử trong cửa sổ `indexDiff` gần nhất (mỗi ngăn chứa tối đa một phần tử).

In [19]:
def exercise_5_contains_nearby_almost_duplicate(nums: List[int], indexDiff: int, valueDiff: int) -> bool:
    # TODO
    pass


# assert exercise_5_contains_nearby_almost_duplicate([1, 2, 3, 1], 3, 0) is True
# assert exercise_5_contains_nearby_almost_duplicate([1, 5, 9, 1, 5, 9], 2, 3) is False
# assert exercise_5_contains_nearby_almost_duplicate([-3, 3], 2, 4) is False

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_bucket` sau khi tự làm.

**Bài 3**
- Mức 1: `exp = 1`; trong khi `max(nums) // exp > 0`: phân phối theo `(x // exp) % 10` vào 10 list (giữ thứ tự), nối lại, `exp *= 10`.

**Bài 4**
- Mức 1: `cnt = [0] * (m + 1)`; với mỗi giá trị `v`, ghi `v` đúng `cnt[v]` lần.

**Bài 5**
- Mức 1: ngăn của `x` là `x // (valueDiff + 1)` (Python `//` làm tròn xuống nên số âm cũng đúng).
- Mức 2: hai số cùng ngăn → chắc chắn cách nhau `<= valueDiff` → trả `True`. Hai số ở ngăn **kề** có thể thỏa → kiểm tra trực tiếp.
- Mức 3: khi `i >= indexDiff`, xóa ngăn của `nums[i − indexDiff]` để cửa sổ chỉ còn `indexDiff` phần tử.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [20]:
_ = trace_bucket([12, 3, 25, 7, 40])

lo = 3, hi = 40, n = 5 → size = 9, số ngăn = 5
   ngăn 0 [3, 12)   : [3, 7] min 3 max 7
   ngăn 1 [12, 21)  : [12] min 12 max 12 | khoảng từ ngăn trước: 12 - 7 = 5
   ngăn 2 [21, 30)  : [25] min 25 max 25 | khoảng từ ngăn trước: 25 - 12 = 13
   ngăn 3 [30, 39)  : rỗng
   ngăn 4 [39, 48)  : [40] min 40 max 40 | khoảng từ ngăn trước: 40 - 25 = 15
→ khoảng lớn nhất: 15


## Đáp án Bài 2

In [21]:
def answer_2_maximum_gap(nums: List[int]) -> int:
    return max_gap_bucket(nums)

assert answer_2_maximum_gap([3, 6, 9, 1]) == 3
assert answer_2_maximum_gap([10]) == 0
assert answer_2_maximum_gap([1, 3, 6, 9, 10, 17]) == 7
print("Bài 2 PASS (giống max_gap_bucket ở mục 5)")

Bài 2 PASS (giống max_gap_bucket ở mục 5)


## Đáp án Bài 3

In [22]:
def answer_3_radix_sort(nums: List[int]) -> List[int]:
    a = list(nums)
    if not a:
        return a
    exp = 1
    while max(a) // exp > 0:
        buckets = [[] for _ in range(10)]
        for x in a:
            buckets[(x // exp) % 10].append(x)     # giữ thứ tự → ổn định
        a = [x for b in buckets for x in b]
        exp *= 10
    return a

assert answer_3_radix_sort([170, 45, 75, 90, 802, 24, 2, 66]) == [2, 24, 45, 66, 75, 90, 170, 802]
assert answer_3_radix_sort([]) == []
assert answer_3_radix_sort([5, 0, 5]) == [0, 5, 5]
for _ in range(1000):
    v = [random.randint(0, 10**6) for _ in range(random.randint(0, 20))]
    assert answer_3_radix_sort(v) == sorted(v)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [23]:
def answer_4_counting_sort(nums: List[int], m: int) -> List[int]:
    cnt = [0] * (m + 1)
    for v in nums:
        cnt[v] += 1
    out = []
    for v, c in enumerate(cnt):
        out.extend([v] * c)
    return out

assert answer_4_counting_sort([3, 1, 2, 3, 0, 1], 3) == [0, 1, 1, 2, 3, 3]
assert answer_4_counting_sort([], 5) == []
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [24]:
def answer_5_contains_nearby_almost_duplicate(nums: List[int], indexDiff: int, valueDiff: int) -> bool:
    w = valueDiff + 1
    buckets = {}                                    # id ngăn -> giá trị (tối đa 1 phần tử mỗi ngăn)
    for i, x in enumerate(nums):
        b = x // w
        if b in buckets:
            return True
        if b - 1 in buckets and x - buckets[b - 1] <= valueDiff:
            return True
        if b + 1 in buckets and buckets[b + 1] - x <= valueDiff:
            return True
        buckets[b] = x
        if i >= indexDiff:
            del buckets[nums[i - indexDiff] // w]
    return False

def brute_220(nums, k, t):
    return any(abs(nums[i] - nums[j]) <= t for i in range(len(nums)) for j in range(i + 1, min(len(nums), i + k + 1)))

assert answer_5_contains_nearby_almost_duplicate([1, 2, 3, 1], 3, 0) is True
assert answer_5_contains_nearby_almost_duplicate([1, 5, 9, 1, 5, 9], 2, 3) is False
assert answer_5_contains_nearby_almost_duplicate([-3, 3], 2, 4) is False
for _ in range(5000):
    nums = [random.randint(-10, 10) for _ in range(random.randint(1, 10))]
    k, t = random.randint(1, 4), random.randint(0, 4)
    assert answer_5_contains_nearby_almost_duplicate(nums, k, t) == brute_220(nums, k, t), (nums, k, t)
print("Bài 5 PASS")

Bài 5 PASS


Vì sao mỗi ngăn chỉ cần một phần tử? Nếu hai phần tử trong cửa sổ rơi vào cùng ngăn thì ta đã trả `True` ngay lúc thêm phần tử thứ hai.

# 14. Tổng kết

> Khoảng lớn nhất ≥ trung bình `(hi − lo)/(n − 1)`. Chia `[lo, hi]` thành các ngăn rộng `size = max(1, (hi − lo)//(n − 1))` → khoảng lớn nhất không nằm trong một ngăn. Chỉ giữ `min`/`max` mỗi ngăn, đáp án là `max(min ngăn sau − max ngăn khác rỗng trước)`. `O(n)` thời gian và bộ nhớ.

---

## Checklist trước khi nộp

- [ ] `n < 2` hoặc mọi số bằng nhau → `0`?
- [ ] `size` chặn dưới bằng 1?
- [ ] Số ngăn `k = (hi − lo) // size + 1`?
- [ ] Bỏ qua ngăn rỗng khi nối các ngăn?